In [1]:
import torch
import numpy as np
import pandas as pd
import os
import gc
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
    pipeline
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

d:\program output\resume project\emotion_detection predefined model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# configuration
CONFIG = {
    'model_name': 'roberta-base',           # Model to use (matches paper)
    'num_epochs': 8,                         # Number of training epochs
    'batch_size': 16,                        # Batch size (reduce if OOM)
    'learning_rate': 2e-5,                   # Learning rate
    'max_length': 128,                       # Max token length
    'warmup_ratio': 0.1,                    # Warmup ratio
    'weight_decay': 0.01,                   # Weight decay
    'seed': 42,                              # Random seed
    'output_dir': '../model/emotion_model_final',   # Where to save model
    'results_dir': '../model/evaluation_results',   # Where to save results
}

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

# Set seed for reproducibility
set_seed(CONFIG['seed'])

print("="*70)
print(" " * 20 + "EMOTION RECOGNITION SYSTEM")
print("="*70)
print(f"\n📋 Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

                    EMOTION RECOGNITION SYSTEM

📋 Configuration:
   model_name: roberta-base
   num_epochs: 8
   batch_size: 16
   learning_rate: 2e-05
   max_length: 128
   warmup_ratio: 0.1
   weight_decay: 0.01
   seed: 42
   output_dir: ../model/emotion_model_final
   results_dir: ../model/evaluation_results


In [4]:
# Load dataset
print("\n" + "="*70)
print("PART 1: LOADING DATASET")
print("="*70)

print("\n📥 Loading dair-ai/emotion dataset...")
dataset = load_dataset("dair-ai/emotion")

print(f"\n✅ Dataset loaded successfully!")
print(f"   Train samples: {len(dataset['train']):,}")
print(f"   Validation samples: {len(dataset['validation']):,}")
print(f"   Test samples: {len(dataset['test']):,}")

# Get label names
label_names = dataset["train"].features["label"].names
print(f"\n📊 Emotion classes: {label_names}")

# Display class distribution
print("\n📊 Class distribution in training set:")
train_labels = dataset["train"]["label"]
for i, name in enumerate(label_names):
    count = train_labels.count(i)
    percentage = count / len(train_labels) * 100
    bar = "█" * int(percentage / 2) + "░" * (50 - int(percentage / 2))
    print(f"   {name:10} : {count:5,} ({percentage:5.1f}%) {bar}")


PART 1: LOADING DATASET

📥 Loading dair-ai/emotion dataset...

✅ Dataset loaded successfully!
   Train samples: 16,000
   Validation samples: 2,000
   Test samples: 2,000

📊 Emotion classes: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

📊 Class distribution in training set:
   sadness    : 4,666 ( 29.2%) ██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
   joy        : 5,362 ( 33.5%) ████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
   love       : 1,304 (  8.2%) ████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
   anger      : 2,159 ( 13.5%) ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
   fear       : 1,937 ( 12.1%) ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
   surprise   :   572 (  3.6%) █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░


In [5]:
# Tokenization
print("\n" + "="*70)
print("PART 2: TOKENIZATION")
print("="*70)

print(f"\n🔤 Loading tokenizer: {CONFIG['model_name']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

def tokenize_function(examples):
    """Tokenize text examples"""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=CONFIG['max_length']
    )

print("\n🔄 Tokenizing datasets...")
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Remove original text column to save memory
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Set format for PyTorch
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

print(f"✅ Tokenization complete!")
print(f"   Max sequence length: {CONFIG['max_length']}")
print(f"   Vocabulary size: {tokenizer.vocab_size:,}")


PART 2: TOKENIZATION

🔤 Loading tokenizer: roberta-base

🔄 Tokenizing datasets...
✅ Tokenization complete!
   Max sequence length: 128
   Vocabulary size: 50,265


In [6]:
# Load model

print("\n" + "="*70)
print("PART 3: LOADING MODEL")
print("="*70)

print(f"\n🤖 Loading model: {CONFIG['model_name']}")
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=6,
    ignore_mismatched_sizes=True
)

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model loaded successfully!")
print(f"   Device: {device}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: {total_params * 4 / 1024 / 1024:.1f} MB")


PART 3: LOADING MODEL

🤖 Loading model: roberta-base


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8670.01it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ Model loaded successfully!
   Device: cuda
   Total parameters: 124,650,246
   Trainable parameters: 124,650,246
   Model size: 475.5 MB


In [14]:
# Trinning setup

training_args = TrainingArguments(
    # Output
    output_dir=CONFIG['output_dir'],
    
    # Training
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=1,
    
    # Optimization
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    
    # Evaluation - CRITICAL: use 'eval_strategy' not 'evaluation_strategy'
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    
    # Logging
    logging_dir=f"{CONFIG['output_dir']}/logs",
    logging_steps=50,
    report_to="none",           # Disable tracking integrations
    
    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,    # Windows compatibility
    dataloader_pin_memory=False,
    
    # Other
    eval_on_start=False,         # Prevent evaluation before training
    seed=CONFIG['seed'],
    disable_tqdm=False,
)

print("\n📋 Training Arguments:")
print(f"   Epochs: {CONFIG['num_epochs']}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Learning rate: {CONFIG['learning_rate']}")
print(f"   eval_strategy: {training_args.eval_strategy}")
print(f"   fp16: {training_args.fp16}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



📋 Training Arguments:
   Epochs: 8
   Batch size: 16
   Learning rate: 2e-05
   eval_strategy: IntervalStrategy.EPOCH
   fp16: True


In [15]:
# Metrics function for trainer
def compute_metrics(eval_pred):
    """Calculate metrics during training"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

In [16]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n✅ Trainer initialized successfully!")


✅ Trainer initialized successfully!


In [17]:
# Trainning

print("\n" + "="*70)
print("PART 5: TRAINING")
print("="*70)

# Calculate training statistics
steps_per_epoch = len(tokenized_dataset["train"]) // CONFIG['batch_size']
total_steps = steps_per_epoch * CONFIG['num_epochs']

print(f"\n📊 Training Statistics:")
print(f"   Training samples: {len(tokenized_dataset['train']):,}")
print(f"   Batch size: {CONFIG['batch_size']}")
print(f"   Steps per epoch: {steps_per_epoch:,}")
print(f"   Total steps: {total_steps:,}")
print(f"   Estimated time: ~{total_steps * 0.1 / 60:.0f} minutes")

# Clear cache before training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"   GPU memory cleared: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB free")

print("\n🚀 Starting training...")
print("-" * 50)

try:
    trainer.train()
    print("\n✅ Training completed successfully!")
except Exception as e:
    print(f"\n❌ Training error: {e}")
    print("\nTroubleshooting tips:")
    print("1. Reduce batch_size to 8")
    print("2. Set fp16=False")
    print("3. Reduce max_length to 64")
    raise


PART 5: TRAINING

📊 Training Statistics:
   Training samples: 16,000
   Batch size: 16
   Steps per epoch: 1,000
   Total steps: 8,000
   Estimated time: ~13 minutes
   GPU memory cleared: 0.50 GB free

🚀 Starting training...
--------------------------------------------------


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.345947,0.276804,0.907500,0.908560
2,0.197733,0.191413,0.930000,0.929522
3,0.146624,0.165169,0.940000,0.940193
4,0.114246,0.158315,0.944500,0.944725
5,0.075193,0.194390,0.937500,0.937477
6,0.085242,0.180156,0.931000,0.931360
7,0.061479,0.244642,0.935000,0.934829


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la


✅ Training completed successfully!


In [18]:
# Save model
print("\n" + "="*70)
print("PART 6: SAVING MODEL")
print("="*70)

# Save model and tokenizer
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

# Save label mapping
import json
label_mapping = {
    'id_to_label': {str(i): label_names[i] for i in range(len(label_names))},
    'label_to_id': {label_names[i]: i for i in range(len(label_names))}
}
with open(f"{CONFIG['output_dir']}/label_mapping.json", 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f"\n✅ Model saved to: {CONFIG['output_dir']}")
print(f"   - model.safetensors")
print(f"   - tokenizer files")
print(f"   - label_mapping.json")


PART 6: SAVING MODEL


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.56s/it]


✅ Model saved to: ../model/emotion_model_final
   - model.safetensors
   - tokenizer files
   - label_mapping.json


In [22]:
# Mannual evaluation
print("\n" + "="*70)
print(" Part 1 MANUAL EVALUATION MATRIX")
print("="*70)

# Load the saved model for evaluation
print("\n🔧 Loading saved model for evaluation...")
eval_model = AutoModelForSequenceClassification.from_pretrained(CONFIG['output_dir'])
eval_model = eval_model.to(device)
eval_model.eval()
print("✅ Model loaded")

# Get test dataset
test_dataset = dataset["test"]
print(f"\n📊 Evaluating on {len(test_dataset)} test samples...")

# Make predictions manually
all_predictions = []
all_labels = []
batch_size = 32

print("\n🔮 Making predictions...")
for i in tqdm(range(0, len(test_dataset), batch_size), desc="Processing batches"):
    batch = test_dataset[i:i+batch_size]
    
    # Tokenize batch
    inputs = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=CONFIG['max_length'],
        return_tensors="pt"
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass
    with torch.no_grad():
        outputs = eval_model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(batch["label"])

# Convert to numpy arrays
y_true = np.array(all_labels)
y_pred = np.array(all_predictions)

print(f"\n✅ Predictions complete!")
print(f"   Total predictions: {len(y_pred)}")
print(f"   Correct predictions: {np.sum(y_true == y_pred)}")
print(f"   Accuracy: {np.sum(y_true == y_pred) / len(y_true):.4f}")


 Part 1 MANUAL EVALUATION MATRIX

🔧 Loading saved model for evaluation...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15281.87it/s]


✅ Model loaded

📊 Evaluating on 2000 test samples...

🔮 Making predictions...


Processing batches: 100%|██████████| 63/63 [00:03<00:00, 18.31it/s]


✅ Predictions complete!
   Total predictions: 2000
   Correct predictions: 1868
   Accuracy: 0.9340


In [ ]:
# calculation Metrics
print("\n" + "="*70)
print("PART 2: CALCULATING METRICS")
print("="*70)

# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n📈 OVERALL METRICS:")
print("-" * 40)
print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1 Score:  {f1:.4f}")

# Per-class metrics
per_class_precision = precision_score(y_true, y_pred, average=None, labels=range(6))
per_class_recall = recall_score(y_true, y_pred, average=None, labels=range(6))
per_class_f1 = f1_score(y_true, y_pred, average=None, labels=range(6))
per_class_support = [np.sum(y_true == i) for i in range(6)]

print("\n📊 PER-CLASS METRICS:")
print("-" * 70)
print(f"{'Emotion':12} {'Precision':>10} {'Recall':>10} {'F1 Score':>10} {'Support':>10}")
print("-" * 70)
for i, name in enumerate(label_names):
    print(f"{name:12} {per_class_precision[i]:>10.4f} {per_class_recall[i]:>10.4f} "
          f"{per_class_f1[i]:>10.4f} {per_class_support[i]:>10}")

# Confusion Matrix
print("\n📊 CONFUSION MATRIX:")
print("-" * 70)
cm = confusion_matrix(y_true, y_pred)

# Print confusion matrix as formatted table
print("\nPredicted →")
print("Actual ↓")
print("-" * 70)

# Header
header = " " * 12 + "".join([f"{name[:8]:>10}" for name in label_names])
print(header)

# Rows
for i, name in enumerate(label_names):
    row = f"{name:>12}" + "".join([f"{cm[i][j]:>10}" for j in range(6)])
    print(row)



PART 2: CALCULATING METRICS

📈 OVERALL METRICS:
----------------------------------------
   Accuracy:  0.9340 (93.40%)
   Precision: 0.9362
   Recall:    0.9340
   F1 Score:  0.9340

📊 PER-CLASS METRICS:
----------------------------------------------------------------------
Emotion       Precision     Recall   F1 Score    Support
----------------------------------------------------------------------
sadness          0.9595     0.9776     0.9685        581
joy              0.9758     0.9295     0.9521        695
love             0.8066     0.9182     0.8588        159
anger            0.9470     0.9091     0.9276        275
fear             0.8629     0.9554     0.9068        224
surprise         0.8302     0.6667     0.7395         66

📊 CONFUSION MATRIX:
----------------------------------------------------------------------

Predicted →
Actual ↓
----------------------------------------------------------------------
               sadness       joy      love     anger      fear  surpr

In [24]:
# Classification report

print("\n" + "="*70)
print("PART 3: ERROR ANALYSIS")
print("="*70)

misclassified_idx = np.where(y_true != y_pred)[0]
print(f"\n📊 Misclassification Statistics:")
print(f"   Total misclassified: {len(misclassified_idx)}/{len(y_true)} ({len(misclassified_idx)/len(y_true)*100:.1f}%)")

# Count errors by class
error_counts = {}
for idx in misclassified_idx:
    true_class = label_names[y_true[idx]]
    pred_class = label_names[y_pred[idx]]
    key = f"{true_class} → {pred_class}"
    error_counts[key] = error_counts.get(key, 0) + 1

# Show top 10 most common errors
print("\n🔍 Most Common Misclassifications:")
print("-" * 40)
sorted_errors = sorted(error_counts.items(), key=lambda x: x[1], reverse=True)
for i, (error, count) in enumerate(sorted_errors[:10]):
    print(f"   {i+1:2}. {error:<25} : {count:3} times")

# Show sample misclassifications
print("\n📝 Sample Misclassified Examples:")
print("-" * 60)
for idx in misclassified_idx[:15]:
    true_label = label_names[y_true[idx]]
    pred_label = label_names[y_pred[idx]]
    text = test_dataset[idx]["text"]
    if len(text) > 70:
        text = text[:67] + "..."
    print(f"   Text: {text}")
    print(f"   True: {true_label} → Predicted: {pred_label}")
    print()


PART 3: ERROR ANALYSIS

📊 Misclassification Statistics:
   Total misclassified: 132/2000 (6.6%)

🔍 Most Common Misclassifications:
----------------------------------------
    1. joy → love                :  34 times
    2. surprise → fear           :  19 times
    3. love → joy                :  13 times
    4. anger → sadness           :  12 times
    5. anger → fear              :  10 times
    6. joy → surprise            :   9 times
    7. sadness → anger           :   8 times
    8. fear → sadness            :   6 times
    9. sadness → fear            :   4 times
   10. joy → sadness             :   4 times

📝 Sample Misclassified Examples:
------------------------------------------------------------
   Text: i felt anger when at the end of a telephone call
   True: anger → Predicted: fear

   Text: i explain why i clung to a relationship with a boy who was in many ...
   True: joy → Predicted: love

   Text: i feel a bit stressed even though all the things i have going on ar..

In [25]:
# Best and wrost perfomances
print("\n" + "="*70)
print("PART 4: BEST AND WORST PERFORMING CLASSES")
print("="*70)

best_idx = np.argmax(per_class_f1)
worst_idx = np.argmin(per_class_f1)

print(f"\n🏆 BEST PERFORMING CLASS:")
print(f"   Emotion: {label_names[best_idx]}")
print(f"   F1 Score: {per_class_f1[best_idx]:.4f}")
print(f"   Precision: {per_class_precision[best_idx]:.4f}")
print(f"   Recall: {per_class_recall[best_idx]:.4f}")

print(f"\n❌ WORST PERFORMING CLASS:")
print(f"   Emotion: {label_names[worst_idx]}")
print(f"   F1 Score: {per_class_f1[worst_idx]:.4f}")
print(f"   Precision: {per_class_precision[worst_idx]:.4f}")
print(f"   Recall: {per_class_recall[worst_idx]:.4f}")


PART 4: BEST AND WORST PERFORMING CLASSES

🏆 BEST PERFORMING CLASS:
   Emotion: sadness
   F1 Score: 0.9685
   Precision: 0.9595
   Recall: 0.9776

❌ WORST PERFORMING CLASS:
   Emotion: surprise
   F1 Score: 0.7395
   Precision: 0.8302
   Recall: 0.6667


In [26]:
# Compare with paper
print("\n" + "="*70)
print("PART 4: COMPARISON WITH PAPER")
print("="*70)

paper_accuracy = 0.88  # 88% from the paper
paper_f1 = 0.88

print(f"\n📄 Paper Results (Pajon-Sanmartin et al., 2025):")
print(f"   Accuracy: {paper_accuracy*100:.1f}%")
print(f"   F1 Score: {paper_f1:.4f}")

print(f"\n🤖 Your Results:")
print(f"   Accuracy: {accuracy*100:.2f}%")
print(f"   F1 Score: {f1:.4f}")

print(f"\n📊 Comparison:")
if accuracy >= paper_accuracy:
    improvement = (accuracy - paper_accuracy) * 100
    print(f"   🎉 You EXCEEDED the paper's results!")
    print(f"   Improvement: +{improvement:.1f}%")
else:
    gap = (paper_accuracy - accuracy) * 100
    print(f"   📝 Gap to paper: {gap:.1f}%")
    print(f"   Tip: Try increasing epochs or using roberta-large")


PART 4: COMPARISON WITH PAPER

📄 Paper Results (Pajon-Sanmartin et al., 2025):
   Accuracy: 88.0%
   F1 Score: 0.8800

🤖 Your Results:
   Accuracy: 93.40%
   F1 Score: 0.9340

📊 Comparison:
   🎉 You EXCEEDED the paper's results!
   Improvement: +5.4%


In [28]:
# Create visualization
print("\n" + "="*70)
print("CREATING VISUALIZATIONS")
print("="*70)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Set style
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("Set2")
    
    # Figure 1: Confusion Matrix Heatmap
    print("\n📊 Creating confusion matrix heatmap...")
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names)
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{CONFIG['results_dir']}/confusion_matrix_heatmap.png", dpi=150)
    plt.close()
    print("   ✅ Saved: confusion_matrix_heatmap.png")
    
    # Figure 2: Per-Class Metrics Bar Chart
    print("📊 Creating per-class metrics bar chart...")
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(label_names))
    width = 0.25
    
    bars1 = ax.bar(x - width, per_class_precision, width, label='Precision', alpha=0.8)
    bars2 = ax.bar(x, per_class_recall, width, label='Recall', alpha=0.8)
    bars3 = ax.bar(x + width, per_class_f1, width, label='F1 Score', alpha=0.8)
    
    ax.set_xlabel('Emotions', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(label_names, rotation=45, ha='right')
    ax.legend(loc='lower right')
    ax.set_ylim(0, 1.1)
    
    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['results_dir']}/per_class_metrics.png", dpi=150)
    plt.close()
    print("   ✅ Saved: per_class_metrics.png")
    
    # Figure 3: Overall Metrics Bar Chart
    print("📊 Creating overall metrics bar chart...")
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    values = [accuracy, precision, recall, f1]
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
    
    plt.figure(figsize=(8, 6))
    bars = plt.bar(metrics, values, color=colors, alpha=0.8)
    plt.ylim(0, 1)
    plt.ylabel('Score', fontsize=12)
    plt.title('Overall Model Performance', fontsize=14, fontweight='bold')
    
    # Add value labels
    for bar, value in zip(bars, values):
        plt.annotate(f'{value:.4f}',
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['results_dir']}/overall_metrics.png", dpi=150)
    plt.close()
    print("   ✅ Saved: overall_metrics.png")
    
    print("\n✅ All visualizations created successfully!")
    
except ImportError as e:
    print(f"\n⚠️ Visualization libraries not available: {e}")
    print("   Install with: pip install matplotlib seaborn")


CREATING VISUALIZATIONS

📊 Creating confusion matrix heatmap...
   ✅ Saved: confusion_matrix_heatmap.png
📊 Creating per-class metrics bar chart...
   ✅ Saved: per_class_metrics.png
📊 Creating overall metrics bar chart...
   ✅ Saved: overall_metrics.png

✅ All visualizations created successfully!
